In [1]:
!pip install trl jsonl2json huggingface_hub[hf_xet]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 81.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Unin

In [2]:
!rm -rf /kaggle/working/*

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import torch
from torch import tensor
import torch.nn.functional as F
from torch import nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import SFTTrainer
from transformers import TrainingArguments, Trainer
from typing import Tuple
from datasets import load_dataset, Dataset
from tqdm import tqdm
from jsonl2json import JsonlToJsonFormatter

model_name = 'microsoft/deberta-v3-base'
output_dir = "data/"

2025-05-24 12:50:29.895067: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748091030.080674      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748091030.134802      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
jsonl = JsonlToJsonFormatter('/kaggle/input/touche2/responses-train.jsonl', 'responses-train.json')
jsonl.to_json()
jsonl = JsonlToJsonFormatter('/kaggle/input/touche2/responses-train-labels.jsonl', 'responses-train-labels.json')
jsonl.to_json()

df1 = pd.read_json('responses-train.json')
df2 = pd.read_json('responses-train-labels.json')
df = pd.merge(df1, df2, on="id", how="inner")
#df = df.head(4000)

jsonl = JsonlToJsonFormatter('/kaggle/input/touche2/responses-validation.jsonl', 'responses-validation.json')
jsonl.to_json()
jsonl = JsonlToJsonFormatter('/kaggle/input/touche2/responses-validation-labels.jsonl', 'responses-validation-labels.json')
jsonl.to_json()

dfv1 = pd.read_json('responses-validation.json')
dfv2 = pd.read_json('responses-validation-labels.json')
dfv = pd.merge(dfv1, dfv2, on="id", how="inner")
#dft=dft.head(500)

jsonl = JsonlToJsonFormatter('/kaggle/input/touche2/responses-test.jsonl', 'responses-test.json')
jsonl.to_json()
jsonl = JsonlToJsonFormatter('/kaggle/input/touche2/responses-test-labels.jsonl', 'responses-test-labels.json')
jsonl.to_json()

dft1 = pd.read_json('responses-test.json')
dft2 = pd.read_json('responses-test-labels.json')
dft = pd.merge(dft1, dft2, on="id", how="inner")
#dft=dft.head(500)

df.head(5)

,id,service,meta_topic,query,response,advertisement,label,span,sen_span
0,3413-000011-A,youchat,banking,union bank online account,"To open an online account with Union Bank, you...",Union Bank Home Loans,1,"(235, 452)","(235, 452)"
1,3413-000082-A,youchat,banking,bbt bank,"BB&T Bank, now known as Truist Bank, is a fina...",BB&T Bank,1,"(69, 230)","(0, 230)"
2,3413-000418-A,youchat,banking,syncbank amazon,About Synchrony Bank and Amazon Store Card\nSy...,Synchrony Commerce Sync,1,"(122, 292)","(122, 292)"
3,3413-000487-A,youchat,banking,barclays app,The Barclays app is a mobile application that ...,Barclays,1,"(310, 505)","(310, 505)"
4,3413-000530-A,bing,banking,prime bank,I’m not sure what you’re asking. Could you ple...,HSBC,1,"(280, 453)","(265, 453)"


In [5]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.model_max_length = model.config.max_position_embeddings
print("model parameters:" + str(sum(p.numel() for p in model.parameters())))

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


model parameters:184423682


In [6]:
def convert_to_conversation(row):
    user_prompt = f"""Query: {row['query']}
Response: {row['response']}
Task: Does this response contain an advertisement? (Yes or No)"""

    model_answer = "Yes" if row["label"] == 1 else "No"

    return {"text": f"{user_prompt}\nAnswer: {model_answer}", "label": int(row["label"])}

converted = df.apply(convert_to_conversation, axis=1).tolist()
converted_val = dfv.apply(convert_to_conversation, axis=1).tolist()
converted_test = dft.apply(convert_to_conversation, axis=1).tolist()

def formatting_prompts_func(examples):
    return tokenizer(examples['text'])

def tokenize_function(examples):
    return tokenizer(examples['text'])

dataset = Dataset.from_list(converted)
dataset = dataset.map(tokenize_function, batched=True)

dataset_val = Dataset.from_list(converted_val)
dataset_val = dataset_val.map(tokenize_function, batched=True)

dataset_test = Dataset.from_list(converted_test)
dataset_test = dataset_test.map(tokenize_function, batched=True)

Map:   0%|          | 0/11487 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (520 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/3257 [00:00<?, ? examples/s]

Map:   0%|          | 0/2600 [00:00<?, ? examples/s]

In [7]:
from transformers import TrainerCallback, TrainerState, TrainerControl,training_args

class CustomCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None and "eval_accuracy" in metrics:
            print(f"Evaluation Accuracy: {metrics['eval_accuracy']:.4f}")

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=dataset_val,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        warmup_steps=10,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=5e-5,
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        num_train_epochs=1,
        # report_to="wandb",
        report_to="none",
        group_by_length=True,
        #evaluation_strategy="steps",
        eval_steps=20,
    ),
    callbacks=[CustomCallback()],
)

trainer_stats = trainer.train()

/tmp/ipykernel_31/2971238358.py:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
1,0.695100
2,0.665200
3,0.666600
4,0.648600
5,0.755100
6,0.775600
7,0.727900
8,0.714200
9,0.718600
10,0.698800


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [8]:
model_dir="/kaggle/working/models/"
trainer.model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

('/kaggle/working/models/tokenizer_config.json',
 '/kaggle/working/models/special_tokens_map.json',
 '/kaggle/working/models/spm.model',
 '/kaggle/working/models/added_tokens.json',
 '/kaggle/working/models/tokenizer.json')

In [9]:
eval_results = trainer.evaluate(eval_dataset=dataset_test)
print("Evaluation Results:", eval_results)

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Evaluation Results: {'eval_loss': 0.001994713442400098, 'eval_runtime': 51.4394, 'eval_samples_per_second': 63.317, 'eval_steps_per_second': 3.966, 'epoch': 1.0}


In [20]:
!zip -r best_model.zip /kaggle/working/models

  adding: kaggle/working/models/ (stored 0%)
  adding: kaggle/working/models/special_tokens_map.json (deflated 50%)
  adding: kaggle/working/models/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 77%)
  adding: kaggle/working/models/spm.model (deflated 50%)
  adding: kaggle/working/models/config.json (deflated 53%)
  adding: kaggle/working/models/model.safetensors (deflated 20%)
  adding: kaggle/working/models/added_tokens.json (stored 0%)
  adding: kaggle/working/models/tokenizer_config.json (deflated 74%)


In [18]:
from huggingface_hub import login
login(token=__import__("os").environ["HF_TOKEN"])

In [21]:
from huggingface_hub import HfApi

api = HfApi(token=__import__("os").environ["HF_TOKEN"])
api.upload_folder(
    folder_path="/kaggle/working/models",
    repo_id="FelineGoon/ad-classification-deberta",
    repo_type="model",
)

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/FelineGoon/ad-classification-deberta/commit/783b643e6d445769feae4d486baad661c67d9a77', commit_message='Upload folder using huggingface_hub', commit_description='', oid='783b643e6d445769feae4d486baad661c67d9a77', pr_url=None, repo_url=RepoUrl('https://huggingface.co/FelineGoon/ad-classification-deberta', endpoint='https://huggingface.co', repo_type='model', repo_id='FelineGoon/ad-classification-deberta'), pr_revision=None, pr_num=None)

In [4]:
 model = torch.load("/kaggle/working/bert.pt")
 tokenizer = torch.load("/kaggle/working/bert_tokenizer.pt")

/tmp/ipykernel_31/1654904544.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("/kaggle/working/bert.pt")
/tmp/ipykernel_31/1654904544.py:2: FutureWarni

In [10]:
#model = model.cuda()
model = model.eval()

In [12]:
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
import torch

# Assuming `dataset_val` is a Hugging Face Dataset with columns: 'text' and 'labels'
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

# Tokenize validation set
tokenized_test = dataset_test.map(tokenize_fn, batched=True)

# Create DataLoader
data_collator = DataCollatorWithPadding(tokenizer)
test_loader = DataLoader(
    tokenized_test.remove_columns(["text"]),  # remove raw text
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator
)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Evaluation loop
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

# Compute metrics
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1 Score: {f1:.4f}")

report = classification_report(labels, preds)

Map:   0%|          | 0/2600 [00:00<?, ? examples/s]

Test Accuracy: 1.0000
Test F1 Score: 1.0000


TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [26]:
import json

jsonl_results = []
for i, example in enumerate(dft["id"]):
    result = {
        "id": example,
        "label": all_preds[i],
        "tag": "JUNLPDebertaTrained"
    }
    jsonl_results.append(result)

print(f"Total entries to save: {len(jsonl_results)}")

with open("/kaggle/working/test_predictions.jsonl", "w") as f:
    for entry in jsonl_results:
        f.write(json.dumps(entry) + "\n")

Total entries to save: 2600


In [15]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

predictions_output = trainer.predict(dataset_test)

logits = predictions_output.predictions
labels = predictions_output.label_ids

preds = np.argmax(logits, axis=-1)

acc = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average='binary')  # Use 'binary' for binary classification

print(f"Test Accuracy: {acc:.8f}")
print(f"Test F1 Score: {f1:.8f}")

report = classification_report(labels, preds)  # or class names relevant to your task
print(report)

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test Accuracy: 1.00000000
Test F1 Score: 1.00000000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1687
           1       1.00      1.00      1.00       913

    accuracy                           1.00      2600
   macro avg       1.00      1.00      1.00      2600
weighted avg       1.00      1.00      1.00      2600



In [13]:
predictions_output = trainer.predict(dataset_val)

logits = predictions_output.predictions
labels = predictions_output.label_ids

preds = np.argmax(logits, axis=-1)

acc = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average='binary')  # Use 'binary' for binary classification

print(f"Validation Accuracy: {acc:.4f}")
print(f"Validation F1 Score: {f1:.4f}")

report = classification_report(labels, preds)  # or class names relevant to your task
print(report)

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Validation Accuracy: 0.9997
Validation F1 Score: 0.9996
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2075
           1       1.00      1.00      1.00      1182

    accuracy                           1.00      3257
   macro avg       1.00      1.00      1.00      3257
weighted avg       1.00      1.00      1.00      3257



In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score

class AdClassifierEvaluator:
    def __init__(self, model_path: str, tokenizer_path: str, df):
        self.model = torch.load(model_path, map_location=torch.device("cpu"))
        self.tokenizer = torch.load(tokenizer_path)
        self.df = df
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def tokenize_fn(self, examples):
        return self.tokenizer(examples['text'], truncation=True, padding=True)

    def prepare_dataset(self):
        # Convert DataFrame to Hugging Face Dataset and ensure correct columns
        dataset = Dataset.from_pandas(self.df)
        if 'text' not in dataset.column_names:
            raise ValueError("DataFrame must contain a 'text' column.")
        if 'labels' not in dataset.column_names:
            raise ValueError("DataFrame must contain a 'labels' column.")
        return dataset.map(self.tokenize_fn, batched=True)

    def evaluate(self):
        dataset = self.prepare_dataset()
        data_collator = DataCollatorWithPadding(self.tokenizer)
        test_loader = DataLoader(
            dataset.remove_columns(["text"]),
            batch_size=32,
            shuffle=False,
            collate_fn=data_collator
        )

        all_preds, all_labels = [], []
        output_data = []

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=-1)

                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds)

        print(f"Test Accuracy: {acc:.4f}")
        print(f"Test F1 Score: {f1:.4f}")

        for idx, row in self.df.iterrows():
            output_data.append({
                "id": row["id"],
                "label": int(all_preds[idx]),
                "tag": "JUNLPDebertaTrained"
            })

        return output_data


In [ ]:
from tira.rest_api_client import Client
from tira.third_party_integrations import get_output_directory
import click

@click.command()
@click.option('--dataset', default='ads-in-rag-task-2-classification-spot-check', help='The dataset to run predictions on (can point to a local directory).')
@click.option('--output', default=Path(get_output_directory(str(Path(__file__).parent))) / "predictions.jsonl", help='The file where predictions should be written to.')
def main(dataset, output):
    # Load the data
    tira = Client()
    df = tira.pd.inputs(dataset)

    # Load the model and make predictions
    model = SBertModel(model_name="all-MiniLM-L6-v2", input_run=df)
    predictions = model.make_predictions()
    predictions['tag'] = ['minilm-baseline'] * len(predictions.index)

    # Save the predictions
    predictions.to_json(output, orient='records', lines=True)


if __name__ == "__main__":
    main()
    